# 미니 GPT를 진짜에 가깝게 — BPE, 학습 안정화, 스케일링

> ⏱ 90분 · T4 GPU 필요 (약 10분) · 선수 지식: [토크나이저](https://yun-sooyong.github.io/dl-study/#30-tokenizer), [미니 GPT](https://yun-sooyong.github.io/dl-study/#31-mini-gpt), [학습 잘 시키는 법](https://yun-sooyong.github.io/dl-study/#24-training-recipes)

**목표:** [미니 GPT](https://yun-sooyong.github.io/dl-study/#31-mini-gpt)는 원리를 보여주는 장난감이었습니다. 이 레슨에서는 실제 LLM 사전학습에 쓰는 요소를 하나씩 붙입니다. **BPE 토크나이저**, **혼합 정밀도**, **그래디언트 클리핑**, **warmup + cosine 스케줄**, 그리고 모델 크기를 바꿔 가며 **스케일링**을 관찰합니다. 이 레슨을 마치면 nanoGPT 같은 실제 사전학습 코드를 읽을 수 있습니다.

## 1. BPE 토크나이저를 붙인다

글자 단위 대신 [토크나이저](https://yun-sooyong.github.io/dl-study/#30-tokenizer) 레슨에서 만든 BPE를 씁니다. 직접 짠 파이썬 버전은 느리므로 Hugging Face의 `tokenizers`(Rust 구현)로 같은 것을 학습합니다.

In [ ]:
import math, time, urllib.request
import torch
from torch import nn
import torch.nn.functional as F
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urllib.request.urlopen(url).read().decode("utf-8")

tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)   # 바이트 수준에서 시작, 공백은 단어 앞에 붙임
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator([text], trainers.BpeTrainer(vocab_size=1024, show_progress=False))

ids = tok.encode(text).ids
print("어휘 크기:", tok.get_vocab_size(), "| 글자 수:", len(text), "→ 토큰 수:", len(ids), f"(글자당 {len(ids)/len(text):.2f} 토큰)")
print(tok.encode("To be, or not to be").tokens)

**코드 읽기**

- `tokenizers` 라이브러리 — `transformers`가 내부에서 쓰는 토크나이저 엔진. 우리가 파이썬으로 짠 BPE와 알고리즘이 같지만 Rust로 되어 있어 100만 글자를 몇 초에 학습합니다.
- `models.BPE()` + `pre_tokenizers.ByteLevel` — GPT-2 방식의 바이트 수준 BPE. `ByteLevel` 전처리는 공백을 특수 문자(`Ġ`)로 바꿔 단어 앞에 붙입니다. 그래서 `" not"`이 하나의 토큰이 됩니다. `decoders.ByteLevel`이 되돌릴 때 이를 다시 공백으로 바꿉니다.
- `vocab_size=1024` — 병합을 약 770번(1024 − 256 바이트). 데이터가 100만 글자뿐이라 어휘를 더 키우면 드문 토큰이 너무 많아집니다. 실제 LLM은 데이터가 수조 토큰이라 어휘 5만~15만이 적당합니다.
- 글자당 약 0.3 토큰 — 같은 텍스트가 **3배 이상 짧은 시퀀스**가 됩니다. 같은 `block_size`로 3배 긴 문맥을 보고, 같은 스텝 수로 3배 많은 글자를 학습하는 셈입니다. 이것이 서브워드 토크나이저를 쓰는 실질적 이유입니다.

In [ ]:
data = torch.tensor(ids)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]
vocab_size = tok.get_vocab_size()

def get_batch(split, block_size, batch_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## 2. 모델: 같은 GPT, 실전용 세부 사항

[미니 GPT](https://yun-sooyong.github.io/dl-study/#31-mini-gpt)와 구조는 같습니다. 달라진 것은 ① 어텐션을 `F.scaled_dot_product_attention` 한 줄로(FlashAttention 등 최적화 커널 사용), ② dropout 추가, ③ 크기를 인자로 받는 것입니다.

In [ ]:
class Attention(nn.Module):
    def __init__(self, d, n_head, dropout):
        super().__init__()
        self.n_head, self.dropout = n_head, dropout
        self.qkv = nn.Linear(d, 3 * d)
        self.proj = nn.Linear(d, d)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0)
        return self.proj(out.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, d, n_head, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = Attention(d, n_head, dropout)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, d=128, n_head=4, n_layer=4, block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(block_size, d)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(d, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight             # 가중치 공유 (weight tying)
        self.apply(self._init)

    @staticmethod
    def _init(m):                                          # GPT-2 방식 초기화: 작은 정규분포
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        logits = self.head(self.ln_f(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.block_size:])
            probs = F.softmax(logits[:, -1] / temperature, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

print(f"{sum(p.numel() for p in GPT().parameters()) / 1e6:.2f}M 파라미터")

**코드 읽기**

- `F.scaled_dot_product_attention(q, k, v, is_causal=True)` — [미니 GPT](https://yun-sooyong.github.io/dl-study/#31-mini-gpt)에서 손으로 짠 네 줄(`q @ k.T / sqrt(d)`, 마스크, softmax, `@ v`)을 PyTorch가 제공하는 최적화 함수로 바꿨습니다. GPU에서는 FlashAttention 커널을 써서 메모리를 시퀀스 길이의 제곱이 아니라 선형으로 쓰고 몇 배 빠릅니다. `is_causal=True`가 마스크를 대신하므로 `register_buffer`도 필요 없습니다. **원리를 아는 사람이 라이브러리를 쓰는 것**이 이 코스의 순서입니다.
- `dropout_p=... if self.training else 0` — 어텐션 가중치에 dropout. `model.train()`/`eval()`에 따라 켜고 끕니다.
- `self.head.weight = self.tok_emb.weight` — 입력 임베딩과 출력층의 가중치를 **공유**합니다. 둘 다 "토큰 ↔ 벡터" 변환이라 같은 행렬을 써도 되고, 파라미터가 `vocab × d`만큼 줄어듭니다. GPT-2를 비롯한 많은 모델이 이렇게 합니다. 어휘가 클수록 절약이 큽니다.
- `self.apply(self._init)` — 모든 `Linear`·`Embedding` 가중치를 표준편차 0.02의 정규분포로 초기화합니다. PyTorch의 `nn.Embedding` 기본 초기화는 표준편차 1이라, 가중치를 공유한 출력층의 로짓이 수십 단위로 커져 초기 손실이 `ln(어휘)`의 10배가 넘게 나옵니다(직접 이 줄을 지우고 초기 loss를 확인해 보세요). 초기화 하나로 학습 초반이 완전히 달라지는 것은 [역전파 직접 구현](https://yun-sooyong.github.io/dl-study/#22-backprop-from-scratch)에서 본 그대로입니다.
- `bias=False` — 최근 LLM들은 Linear의 편향을 대부분 뺍니다(LayerNorm이 있어 불필요하고 약간 빠름).
- 크기 인자 `d, n_head, n_layer` — 아래 스케일링 실험에서 이 셋을 바꿉니다. 이 세 숫자와 `vocab_size`, `block_size`가 트랜스포머 "크기"의 전부입니다. GPT-2 small은 `d=768, n_head=12, n_layer=12`, GPT-3는 `d=12288, n_head=96, n_layer=96`입니다.

## 3. 학습 함수: 실전 4종 세트

In [ ]:
def train(model, steps, block_size=256, batch_size=32, lr=6e-4, warmup=100, log_every=100):
    model.to(device)
    decay = [p for p in model.parameters() if p.dim() >= 2]      # 행렬(가중치)에만 weight decay
    no_decay = [p for p in model.parameters() if p.dim() < 2]    # 편향·LayerNorm은 제외
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
                            lr=lr, betas=(0.9, 0.95))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    hist, t0, tokens = [], time.time(), 0
    for step in range(steps + 1):
        if step % log_every == 0:
            model.eval()
            with torch.no_grad():
                val = sum(model(*get_batch("val", block_size, batch_size))[1].item() for _ in range(10)) / 10
            model.train()
            hist.append((step, val))
            print(f"step {step:5d}  val loss {val:.3f}  lr {sched.get_last_lr()[0]:.2e}  {tokens / (time.time() - t0 + 1e-9):,.0f} tok/s")
        x, y = get_batch("train", block_size, batch_size)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):   # 혼합 정밀도
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)                          # 그래디언트 클리핑
        scaler.step(opt); scaler.update(); sched.step()
        tokens += x.numel()
    return hist

model = GPT()
hist = train(model, steps=1500)

**코드 읽기**

- **weight decay를 행렬에만** — `p.dim() >= 2`인 파라미터(Linear·Embedding 가중치)에만 0.1을 주고, 편향과 LayerNorm(1차원)에는 주지 않습니다. 편향까지 0으로 당기면 손해만 있기 때문입니다. nanoGPT·GPT-2·Llama 모두 이 규칙을 씁니다. `weight_decay=0.1`은 [학습 잘 시키는 법](https://yun-sooyong.github.io/dl-study/#24-training-recipes)보다 큰 값인데, LLM 사전학습의 관례입니다.
- `betas=(0.9, 0.95)` — Adam의 두 번째 모멘텀 계수를 기본값 0.999 대신 0.95로. 기울기 크기의 추정을 더 빨리 갱신해 LLM 학습이 안정적이라는 경험칙(GPT-3 논문)입니다.
- `LambdaLR`의 람다 — `min(1, (s+1)/warmup)`은 처음 100스텝 동안 학습률을 0에서 선형으로 올리고(warmup), `0.5 * (1 + cos(π s / steps))`는 그 뒤 코사인 곡선을 따라 0까지 내립니다. 두 곱이 "warmup + cosine decay". 왜 warmup인가: 학습 초반에는 Adam의 기울기 통계가 아직 부정확해서 큰 학습률로 시작하면 발산하기 쉽습니다.
- `torch.autocast(dtype=torch.float16)` — **혼합 정밀도**. forward의 행렬곱을 16비트로 계산해 메모리를 절반, 속도를 2~3배로. 가중치와 옵티마이저 상태는 32비트로 유지해 정밀도를 지킵니다. T4는 bf16을 지원하지 않아 fp16을 쓰고, fp16은 표현 범위가 좁아 작은 기울기가 0이 되는 것을 막기 위해 `GradScaler`가 손실을 크게 곱했다가(`scale`) 갱신 전에 되돌립니다(`unscale_`). A100 이상에서는 `bfloat16`을 쓰고 스케일러가 필요 없습니다. CPU에서는 자동으로 꺼집니다(`enabled=use_amp`).
- `clip_grad_norm_(model.parameters(), 1.0)` — 전체 기울기 벡터의 길이가 1.0을 넘으면 1.0으로 줄입니다. 가끔 나오는 이상한 배치가 거대한 기울기를 만들어 학습을 망치는 것(loss 스파이크)을 막습니다. LLM 학습에서 거의 예외 없이 씁니다. `unscale_` **뒤에** 해야 스케일러가 곱한 값이 아닌 진짜 기울기의 길이를 재게 됩니다.
- `zero_grad(set_to_none=True)` — 기울기를 0으로 채우는 대신 메모리를 해제해 약간 빠릅니다.
- `tok/s` 출력 — 처리량. 실제 사전학습에서 가장 중요한 공학 지표입니다. "이 GPU로 1B 토큰을 학습하려면 며칠 걸리나"를 여기서 계산합니다.

## 4. 스케일링: 크기를 바꿔 가며 같은 실험

In [ ]:
import matplotlib.pyplot as plt

configs = {"tiny (0.2M)": dict(d=64, n_head=2, n_layer=2), "small (1M)": dict(d=128, n_head=4, n_layer=4), "medium (5M)": dict(d=256, n_head=8, n_layer=6)}
results = {}
for name, cfg in configs.items():
    torch.manual_seed(0)
    m = GPT(**cfg)
    print(f"\n== {name}: {sum(p.numel() for p in m.parameters()) / 1e6:.2f}M 파라미터")
    results[name] = train(m, steps=1500, log_every=300)

for name, h in results.items():
    plt.plot([s for s, _ in h], [v for _, v in h], marker="o", label=name)
plt.xlabel("step"); plt.ylabel("val loss"); plt.legend(); plt.title("same steps, different model sizes"); plt.show()

**코드 읽기**

- 같은 데이터, 같은 스텝 수, 같은 학습률로 **모델 크기만** 바꿉니다. 한 번에 하나만 바꾸는 실험 원칙입니다.
- 결과에서 볼 것: ① 큰 모델이 같은 스텝에서 더 낮은 손실에 도달합니다. ② 그러나 tok/s가 낮아 같은 **시간**에는 덜 학습합니다. ③ 데이터가 100만 글자뿐이라 medium은 뒤로 갈수록 val loss가 정체되거나 올라갑니다(과적합). 이 세 관찰이 스케일링 법칙의 핵심입니다: **손실은 파라미터 수·데이터 양·계산량이 함께 커질 때 예측 가능하게 내려간다.**
- Chinchilla 법칙 — 계산량이 정해져 있을 때 최적 데이터 양은 파라미터의 약 20배 토큰입니다. 우리 medium(5M)이 최적으로 학습하려면 1억 토큰이 필요한데 데이터는 30만 토큰뿐입니다. 300배 부족한 데이터로 큰 모델을 돌리면 과적합하는 것이 당연합니다. 실제 LLM들이 수조 토큰을 모으는 이유입니다.

## 5. 생성

In [ ]:
model.eval()
start = torch.tensor([tok.encode("ROMEO:").ids], device=device)
print(tok.decode(model.generate(start, 200)[0].tolist()))

[미니 GPT](https://yun-sooyong.github.io/dl-study/#31-mini-gpt)보다 단어 철자가 훨씬 안정적입니다. 글자를 하나씩 조립할 필요 없이 토큰(단어 조각) 단위로 생성하기 때문입니다.

## 6. 여기서 진짜 사전학습까지

이 코드와 GPT-2 사전학습 코드(nanoGPT)의 차이는 크기와 데이터뿐입니다. 남은 간극은 **공학**입니다.

| 요소 | 이 레슨 | 실제 사전학습 |
|---|---|---|
| 데이터 | 셰익스피어 100만 글자 | 웹·책·코드 수조 토큰. 중복 제거, 품질 필터링, 언어·도메인 비율 조정(data mix)이 성능의 절반 |
| 토크나이저 | 어휘 1024 | 어휘 3만~15만, 다국어·코드·숫자 처리 규칙 |
| 모델 | 3M | 1B~1T. 구조는 같고 RoPE, RMSNorm, SwiGLU, GQA 같은 개선이 추가됨 |
| 학습 | 1500스텝, GPU 1개 | 수십만 스텝, GPU 수백~수천 개를 묶어 병렬 학습(data/tensor/pipeline parallel) |
| 안정성 | 클리핑, warmup | + 체크포인트에서 재시작, loss 스파이크 감지, 학습률 재조정, 하드웨어 고장 대응 |
| 정밀도 | fp16 autocast | bf16, 일부 fp8 |

직접 해 보고 싶다면: Karpathy의 **nanoGPT**(이 레슨 코드의 원형, GPT-2 124M을 8×A100으로 4일에 재현), **llm.c**(같은 것을 C/CUDA로). 두 저장소를 읽으면 이 레슨의 모든 줄이 어디에 대응하는지 보일 것입니다.

## 핵심 정리

- BPE 토크나이저는 시퀀스를 3배 이상 짧게 만들어 같은 자원으로 더 긴 문맥·더 많은 데이터를 학습하게 합니다.
- 실전 학습의 4종 세트: **AdamW(행렬만 decay, β₂=0.95) · warmup+cosine · 그래디언트 클리핑 · 혼합 정밀도.**
- `F.scaled_dot_product_attention`은 직접 짠 어텐션과 같은 계산을 빠르고 메모리 적게 합니다.
- 큰 모델은 스텝당 더 잘 배우지만 느리고 데이터를 더 요구합니다. 파라미터·데이터·계산량이 **함께** 커져야 합니다(Chinchilla: 토큰 ≈ 20 × 파라미터).
- 실제 사전학습과의 차이는 원리가 아니라 규모와 공학입니다.

## 스스로 점검

답을 머릿속으로 먼저 말해 본 뒤 펼쳐 보세요.

<details><summary>Q1. 혼합 정밀도에서 fp16에는 GradScaler가 필요하고 bf16에는 필요 없는 이유는?</summary>

fp16은 지수 범위가 좁아(최소 약 6e-8) 작은 기울기가 0으로 사라집니다. 손실을 크게 곱해 기울기를 표현 범위 안으로 올렸다가 갱신 전에 되돌리는 것이 GradScaler입니다. bf16은 fp32와 같은 지수 범위를 가져(정밀도만 낮음) 이 문제가 없습니다.

</details>

<details><summary>Q2. 그래디언트 클리핑을 <code>unscale_</code> 전에 하면 무엇이 잘못되나요?</summary>

스케일러가 손실에 곱한 큰 값(예: 65536)이 기울기에 그대로 들어 있어, 기울기 길이가 항상 1.0을 넘는 것으로 보여 모든 스텝에서 기울기가 잘려 버립니다. 실제 기울기 길이를 재려면 먼저 되돌려야 합니다.

</details>

<details><summary>Q3. 같은 시간 예산에서 tiny와 medium 중 어느 것이 더 낮은 손실에 도달할까요?</summary>

데이터가 충분하다면 처음에는 tiny가 빨리 내려가지만 결국 medium이 더 낮은 곳까지 갑니다(tiny는 표현력 한계에 도달). 데이터가 부족하면 medium이 과적합해 tiny와 비슷하거나 나쁠 수 있습니다. 답은 "데이터 양에 달렸다"이고, 그것이 Chinchilla 법칙의 요점입니다.

</details>

<details><summary>Q4. 가중치 공유(weight tying)가 파라미터를 얼마나 줄이나요? GPT-2 small(어휘 50257, d=768)의 경우 계산해 보세요.</summary>

`50257 × 768 ≈ 3860만`. GPT-2 small 전체(1.24억)의 약 31%입니다. 어휘가 큰 모델일수록 절약이 큽니다.

</details>

## 직접 고쳐보기

1. `clip_grad_norm_` 줄을 주석 처리하고 `lr=3e-3`으로 올려 보세요. loss 스파이크나 NaN이 나오나요? 클리핑을 되돌리면 어떻게 되나요?
2. warmup을 0으로(`warmup=1`) 하고 같은 학습률로 학습시켜 보세요. 초반 loss 곡선이 어떻게 다른가요?
3. `vocab_size`를 256(사실상 바이트 단위)과 4096으로 바꿔 토크나이저를 다시 학습하고, 같은 스텝 수로 GPT를 학습시켜 **글자당 손실**(val loss × 글자당 토큰 수)로 비교해 보세요. 토큰 단위 손실은 어휘가 다르면 직접 비교할 수 없습니다.
4. `block_size`를 64와 512로 바꿔 보세요. tok/s와 val loss가 어떻게 변하나요? 어텐션 비용이 길이의 제곱인 것을 tok/s에서 확인할 수 있나요?
5. **한국어로:** `text`를 한국어 텍스트(위키백과 문서 수십 개, 공개 소설 등 최소 100만 글자)로 바꾸고 토크나이저부터 다시 학습하세요. 조사·어미가 토큰이 되는지, 생성 결과가 문법적으로 그럴듯한지 보세요.
6. (도전) 세 크기 모델을 **같은 시간**(예: 각 2분) 동안 학습시켜 비교하세요. 시간 기준에서는 어느 크기가 이기나요? `train`에 시간 제한을 추가해야 합니다.
7. (도전) nanoGPT 저장소의 `model.py`를 열어 이 레슨의 `GPT`와 한 줄씩 대응시켜 보세요. 다른 부분이 무엇인지 목록을 만드세요.

<details><summary>힌트와 예상 결과 — 먼저 스스로 해 본 뒤 펼치세요</summary>

1. 클리핑 없이 `lr=3e-3`이면 수백 스텝 안에 loss가 갑자기 튀거나 `nan`이 됩니다. 클리핑을 되돌리면 같은 학습률에서도 대체로 버팁니다(완전히 안전하지는 않음). 클리핑은 "가끔 오는 이상한 배치"에 대한 보험입니다.
2. warmup이 없으면 처음 몇십 스텝에서 loss가 잠깐 오르거나 초반 곡선이 거칠고, 최종 loss가 약간 높게 끝나는 경우가 많습니다. 작은 모델에서는 차이가 작고, 큰 모델일수록 warmup 없이는 발산합니다.
3. 토큰 단위 val loss는 어휘 256이 가장 낮고 4096이 가장 높게 보이지만, 글자당 손실로 환산하면 1024~4096이 더 좋습니다. 어휘가 클수록 한 토큰이 더 많은 글자를 담아 예측이 어려워지는 것뿐입니다. 비교는 항상 같은 단위로.
4. `block_size=64`: tok/s가 높고 loss는 조금 높음(문맥 부족). 512: tok/s가 절반 이하로 떨어지고 loss는 약간 낮음. 어텐션 비용이 T²이므로 길이 8배에 어텐션 계산은 64배입니다(전체 시간은 MLP 몫이 있어 그보다 덜 늘어남).
5. 한국어는 바이트 수준 BPE에서 글자당 토큰 수가 영어보다 높게 시작하고, 어휘 1024로는 조사·어미가 잘 안 잡힙니다. `vocab_size`를 4000~8000으로 올리고 데이터를 충분히 준비하면 "~습니다", "~에서" 같은 토큰이 생깁니다.
6. 2분 기준에서는 tiny가 스텝을 10배 이상 돌아 small과 비슷하거나 앞서고, medium은 스텝이 부족해 뒤집니다. 시간·계산량이 고정이면 "적당한 크기"가 최적이라는 것이 스케일링 법칙의 실용적 결론입니다.
7. 다른 점: nanoGPT는 `bias` 옵션, Flash 여부 판별, `_init_weights`의 잔차 층 특별 초기화(`0.02/√(2·n_layer)`), 옵티마이저 fused 옵션, `estimate_mfu`(하드웨어 활용률 계산) 등. 구조 자체는 같습니다.

</details>